# GraphRAG — Full Test Notebook (v3)
Run top to bottom. Uses the real `knowledge_graph.gml`.

> **Required:** `outputs/graphs/knowledge_graph.gml` must exist in your Drive folder

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/GraphRAG'

if not os.path.exists(PROJECT_ROOT):
    raise FileNotFoundError(f'Not found: {PROJECT_ROOT}. Upload GraphRAG folder to MyDrive first.')

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Files:', sorted(os.listdir('.')))
print('\n✅ Cell 1 passed — Drive mounted')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────
# Re-run every new Colab session. Safe to skip if already installed.
import subprocess, sys

def install(packages, label):
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + packages, capture_output=True, text=True)
    print(f'{'✅' if result.returncode == 0 else '❌'} {label}')
    if result.returncode != 0: print(result.stderr[-500:])

install(['pyyaml', 'python-dotenv'], 'Core utilities')
install(['tiktoken'],                'Tokenizer')
install(['google-generativeai'],     'Gemini API')
install(['groq'],                    'Groq API')
install(['networkx'],                'NetworkX')
install(['chromadb'],                'ChromaDB')
install(['pymupdf'],                 'PyMuPDF')
install(['streamlit', 'pyvis'],     'Streamlit + Pyvis')
install(['spacy'],                   'spaCy')

r = subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], capture_output=True, text=True)
print('✅ spaCy model' if r.returncode == 0 else '❌ spaCy model: ' + r.stderr[-200:])

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'graspologic'], capture_output=True, text=True)
if r.returncode == 0:
    print('✅ graspologic')
else:
    install(['leidenalg', 'igraph'], 'leidenalg (fallback)')

print('\n--- Done. Re-run once if any ❌ ---')

In [ ]:
# ── Cell 3: Verify all imports ────────────────────────────────
failed = []
for module, pkg in [('yaml','pyyaml'),('dotenv','python-dotenv'),('spacy','spacy'),
                    ('tiktoken','tiktoken'),('google.generativeai','google-generativeai'),
                    ('groq','groq'),('networkx','networkx'),('chromadb','chromadb'),('fitz','pymupdf')]:
    try:
        __import__(module); print(f'  ✅ {pkg}')
    except ImportError as e:
        print(f'  ❌ {pkg} → {e}'); failed.append(pkg)

try:
    import graspologic; print('  ✅ graspologic')
except ImportError:
    try:
        import leidenalg; print('  ✅ leidenalg (fallback)')
    except ImportError:
        print('  ❌ No Leiden backend'); failed.append('leiden')

try:
    import spacy; spacy.load('en_core_web_sm'); print('  ✅ spaCy model')
except OSError:
    print('  ❌ spaCy model missing'); failed.append('spacy-model')

print()
if failed: print(f'❌ {len(failed)} missing: {failed} — go back to Cell 2')
else: print('✅ Cell 3 passed — all imports OK')

In [ ]:
# ── Cell 4: Set API keys + switch to Groq ─────────────────────
# Gemini free tier has quota issues — using Groq instead.
# Get free key: console.groq.com → API Keys → Create

GROQ_API_KEY   = 'gsk_paste_your_groq_key_here'
GEMINI_API_KEY = 'paste_your_gemini_key_here'

if GROQ_API_KEY == 'gsk_paste_your_groq_key_here':
    raise ValueError('Replace GROQ_API_KEY with your actual key before running.')

import os
with open(f'{PROJECT_ROOT}/.env', 'w') as f:
    f.write(f'GEMINI_API_KEY={GEMINI_API_KEY}\nGROQ_API_KEY={GROQ_API_KEY}\n')
os.environ['GROQ_API_KEY']   = GROQ_API_KEY
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

from src.config import cfg
cfg['llm']['generation_provider'] = 'groq'
cfg['llm']['generation_model']    = 'llama-3.3-70b-versatile'
cfg['llm']['extraction_provider'] = 'groq'
cfg['llm']['extraction_model']    = 'llama-3.3-70b-versatile'

print('Provider: Groq — llama-3.3-70b-versatile')
print('\n✅ Cell 4 passed — keys saved, Groq active')

In [ ]:
# ── Cell 5: Test config loader ────────────────────────────────
from src.config import cfg, GROQ_API_KEY as GK, GRAPH_PATH

print('Chunk size:', cfg['chunking']['chunk_size'])
print('LLM provider:', cfg['llm']['generation_provider'])
print('Graph path:', GRAPH_PATH)
print('Graph file exists:', GRAPH_PATH.exists())
print('Groq key loaded:', bool(GK))

assert cfg['chunking']['chunk_size'] == 600
assert bool(GK), 'Groq key not loaded — check Cell 4'

if not GRAPH_PATH.exists():
    print('\n⚠️  knowledge_graph.gml not found — upload it to outputs/graphs/ before Cell 10')
else:
    print('\n✅ knowledge_graph.gml found')

print('\n✅ Cell 5 passed — config OK')

In [ ]:
# ── Cell 6: Test chunker with real NLP content ────────────────
# Using NLP-themed text so LOCAL queries in Cell 12 return meaningful results
from src.chunker import chunk_document, split_into_sentences

sample_text = """
The Transformer architecture was introduced in Attention is All You Need by Vaswani et al. in 2017.
It relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions.
Self-attention allows each position in the sequence to attend to all positions in the previous layer.
Multi-head attention runs multiple attention functions in parallel across different representation subspaces.
Scaled dot-product attention computes weights as softmax of QK transpose divided by square root of d_k.

BERT, Bidirectional Encoder Representations from Transformers, was introduced by Devlin et al. in 2018.
BERT is pre-trained using masked language modeling where 15 percent of tokens are masked and predicted.
Fine-tuning BERT on downstream tasks achieves state-of-the-art results across many NLP benchmarks.
RoBERTa improves on BERT by removing next sentence prediction and training with more data.

GPT uses only the Transformer decoder and is pre-trained with causal language modeling.
GPT-3 with 175 billion parameters demonstrated strong in-context learning across diverse NLP tasks.
Chain-of-thought prompting encourages models to produce intermediate reasoning steps before the final answer.

Retrieval-Augmented Generation combines parametric LLM knowledge with non-parametric retrieval.
RAG retrieves relevant passages using dense vector similarity search over a vector database.
GraphRAG extends RAG by constructing a knowledge graph from extracted entities and relationships.
The Leiden algorithm detects communities of related entities within the knowledge graph.
Community summaries are generated using an LLM to capture thematic content of each community.
A map-reduce framework generates partial answers from community summaries and aggregates them.
The hybrid query router classifies each query as local, global, or hybrid to select the right pipeline.
"""

sentences = split_into_sentences(sample_text)
print(f'Sentences: {len(sentences)}')

chunks = chunk_document(sample_text, doc_id='nlp_overview')
print(f'Chunks: {len(chunks)}')
for c in chunks:
    print(f"  [{c['chunk_index']}] tokens={c['token_count']}  sentences={c['sentences']}")

assert len(chunks) >= 1
assert all(c['token_count'] <= 650 for c in chunks)
print('\n✅ Cell 6 passed — chunker works')

In [ ]:
# ── Cell 7: Test LLM client ───────────────────────────────────
from src.llm_client import LLMClient

llm = LLMClient(purpose='generation')
print('Provider:', llm.provider, '| Model:', llm.model)

response = llm.generate('In one sentence, what is a knowledge graph?')
print('Response:', response)

assert len(response) > 10, 'Response too short — check your API key'
print('\n✅ Cell 7 passed — LLM works')

In [ ]:
# ── Cell 8: Test vector store ─────────────────────────────────
from src.vector_store import VectorStore

vs = VectorStore()
print('Initial stats:', vs.stats())

vs.build_index(chunks)
print('After indexing:', vs.stats())

results = vs.query('how does self-attention work in transformers')
print(f'\nQuery returned {len(results)} results')
for r in results[:2]:
    print(f"  score={r['score']}  text: {r['text'][:100]}...")

assert len(results) > 0
print('\n✅ Cell 8 passed — vector store works')

In [ ]:
# ── Cell 9: Test router ───────────────────────────────────────
from src.router import QueryRouter

router = QueryRouter(llm=llm)

test_queries = [
    ('What datasets were used in the BERT paper?',     'LOCAL'),
    ('What are the main themes across all papers?',    'GLOBAL'),
    ('Explain attention and its role in the field',    'HYBRID'),
]

print('Router results:')
for query, expected in test_queries:
    d = router.route(query)
    match = '✅' if d.route_type.value == expected else '⚠️ '
    print(f"  {match} [{d.route_type.value:6s}] conf={d.confidence:.2f}  '{query[:45]}'")

print('\n(⚠️  = router disagreed — not necessarily wrong)')
print('\n✅ Cell 9 passed — router runs')

In [ ]:
# ── Cell 10: Load real KG + community detection ───────────────
import networkx as nx
from src.community_detection import CommunityDetector
from src.config import GRAPH_PATH

if not GRAPH_PATH.exists():
    raise FileNotFoundError(
        f'knowledge_graph.gml not found at {GRAPH_PATH}\n'
        'Steps:\n'
        '  1. Download knowledge_graph.gml from the chat\n'
        '  2. Upload to MyDrive/GraphRAG/outputs/graphs/\n'
        '  3. Re-run this cell'
    )

cd = CommunityDetector()
cd.load_graph()

print(f'Nodes: {cd.graph.number_of_nodes()}')
print(f'Edges: {cd.graph.number_of_edges()}')

degrees = sorted(cd.graph.degree(), key=lambda x: x[1], reverse=True)[:8]
print('\nTop entities by connections:')
for name, deg in degrees:
    print(f'  {deg:3d}  {name}')

cd.detect_all_levels()

print('\nCommunity stats:')
for res, stats in cd.stats().items():
    print(f'  Resolution {res}: {stats}')

members_1 = cd.get_community_members(1.0)
print(f'\nCommunities at resolution=1.0 ({len(members_1)} total):')
for cid, members in sorted(members_1.items(), key=lambda x: -len(x[1])):
    print(f'  Community {cid} ({len(members)} nodes): {members[:4]}{" ..." if len(members)>4 else ""}')

assert len(members_1) >= 2
print('\n✅ Cell 10 passed — real graph loaded and communities detected')

In [ ]:
# ── Cell 11: Community summarizer (largest community) ─────────
from src.community_summarizer import CommunitySummarizer

cs = CommunitySummarizer(detector=cd, llm=llm)

# Pick the largest community for a richer summary
largest_comm_id = max(members_1, key=lambda x: len(members_1[x]))
largest_members = members_1[largest_comm_id]

print(f'Summarizing community {largest_comm_id} ({len(largest_members)} members)')
print(f'Members: {largest_members}')

summary = cs._summarize_community(largest_comm_id, largest_members, resolution=1.0)

print('\nSummary preview:')
print(summary['summary'][:400], '...')

assert len(summary['summary']) > 50
print('\n✅ Cell 11 passed — summarizer works on real graph')

In [ ]:
# ── Cell 12: Full end-to-end test ─────────────────────────────
# Re-initializes all objects — safe after kernel restart
from src.graph_query import GraphQueryEngine
from src.query_engine import QueryEngine
from src.vector_store import VectorStore
from src.router import QueryRouter
from src.llm_client import LLMClient

llm    = LLMClient(purpose='generation')
vs     = VectorStore()
router = QueryRouter(llm=llm)

# Re-index if empty (e.g. after session restart)
if vs.stats()['total_chunks'] == 0:
    print('Vector store empty — re-indexing...')
    vs.build_index(chunks)
print('Vector store chunks:', vs.stats()['total_chunks'])

print('\nGenerating all community summaries...')
cs.summarize_all(resolution=1.0)

graph_engine = GraphQueryEngine(summarizer=cs, llm=llm)
engine = QueryEngine(vector_store=vs, graph_engine=graph_engine, router=router, llm=llm)

# Test all 3 route types
test_cases = [
    ('What is self-attention and how does it work?',        'LOCAL'),
    ('What are the main research themes in this corpus?',   'GLOBAL'),
    ('Explain BERT and how it fits into the NLP landscape', 'HYBRID'),
]

print('\nEnd-to-end results:')
print('=' * 60)
for query, forced_route in test_cases:
    result = engine.query(query, force_route=forced_route)
    print(f'\n[{result["route"]}] {query}')
    print(f'Answer: {result["answer"][:200]}...')
    assert result['answer'], f'Empty answer for {forced_route}'

print('\n✅ Cell 12 passed — full pipeline works on real graph')

In [ ]:
# ── Cell 13: Summary ──────────────────────────────────────────
print('=' * 50)
print('ALL TESTS PASSED')
print('=' * 50)
print(f"""
Graph stats:
  Nodes          : {cd.graph.number_of_nodes()}
  Edges          : {cd.graph.number_of_edges()}
  Communities    : {len(members_1)} (at resolution=1.0)

Verified:
  Cell 5  Config loader
  Cell 6  Sentence-aware chunker
  Cell 7  LLM client (Groq)
  Cell 8  Vector store (ChromaDB)
  Cell 9  Hybrid query router
  Cell 10 Community detection on real graph
  Cell 11 Community summarizer on real graph
  Cell 12 Full end-to-end pipeline on real graph

When Person A shares the extracted graph:
  Replace outputs/graphs/knowledge_graph.gml and re-run cells 10-12.

Next: streamlit run app.py
""")